# Continuous bath heating – energy demand model

*Thermal activation solid products › Continuous bath heating* (manuscript Fig. 1)

**Examples** (Fig. 1). All use the same model; the examples change nothing in the equations.
| Process | Products |
|---|---|
| Galvanization | steel sheets and coils, wires, pipes and tubing |
| Washing | cotton and synthetic fabrics, denim, nonwoven fabrics |
| Boiling large scale | vegetables, precooked pasta, meat and sea food, fruits for freezing |

A bath is kept at $T_{bath}$. Its heat demand is the product dipped into it plus the losses through the walls and from the open surface by convection and radiation (manuscript section *Continuous bath heating*, Fig. 6). The bath takes all this heat at $T_{bath}$.

$$\dot Q_{product}=\dot m_{product}\,c_{p,product}\,(T_{bath}-T_{product,in})\quad(15)\qquad \dot Q_{trans}=U_{bath}\,A_{wall}\,(T_{bath}-T_{ambient})\quad(16)$$
$$\dot Q_{surface}=\alpha\,A_{surface}\,(T_{bath}-T_{ambient})+\varepsilon\,\sigma\,A_{surface}\,(T_{bath}^4-T_{ambient}^4)\quad(17)$$

This notebook holds no formulas of its own: it imports **`model.py`**, the model the app runs.

1. Example calculation
2. Figures and simplified P&ID (saved as `pid.svg`)
3. Validation against the manuscript and consistency checks
4. Comparison with an existing implementation

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, FancyBboxPatch, Polygon
from IPython.display import display, Markdown

try:
    import process_models
except ImportError:            # not installed: use the copy in this repository
    import sys
    sys.path.insert(0, str(Path.cwd().parents[1]))   # .../process_models (the folder holding the package)

from process_models import PROCESSES, run_process
from process_models.continuous_bath_heating.model import MODEL, calculate

KEY = MODEL.key

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})


def show(res, audit):
    """Input table, results and stream data of one run."""
    display(Markdown("**Inputs** (required / optional, source of each value)"))
    display(pd.DataFrame(audit))
    for w in res.warnings:
        print("WARNING:", w)
    display(Markdown("**Results**"))
    display(pd.DataFrame(res.summary()))
    display(Markdown("**Stream data** (what the app hands to the pinch analysis)"))
    display(pd.DataFrame(res.stream_table()))


print(f"{MODEL.name} v{MODEL.version} | processes found by the app: {list(PROCESSES)}")

## 1  Example calculation
Hot-dip bath for steel parts. The values are **illustrative, not from the manuscript** (it gives no inputs for its
case): 20 t/h steel (cp 0.5 kJ/(kg K)) from 20 °C into a bath at 170 °C, 20 m² surface, 60 m² walls.

In [ ]:
BATH = dict(product_flow_kg_h=20000, cp_product=0.5, T_product_in=20, T_bath=170, A_surface=20, A_wall=60,
            U_bath=0.5, T_amb=20, alpha=10, utility_flow_kg_h=30000, T_utility_in=200)
result, audit = run_process(KEY, **BATH)
show(result, audit)

## 2  Figures and simplified P&ID

In [ ]:
def plot_demand_breakdown(res):
    """Every kW quantity of the results table; totals dark."""
    rows = [r for r in res.summary() if r["Unit"] == "kW"][::-1]
    fig, ax = plt.subplots(figsize=(9, 0.45 * len(rows) + 1.4))
    colors = ["#333333" if r["Quantity"].split(" (")[0].isupper() else "#4c78a8" for r in rows]
    bars = ax.barh([r["Quantity"] for r in rows], [r["Value"] for r in rows], color=colors)
    for b, r in zip(bars, rows):
        ax.annotate(f"{r['Value']:,.1f} kW", (b.get_width(), b.get_y() + b.get_height() / 2),
                    xytext=(4, 0), textcoords="offset points", va="center", fontsize=8)
    ax.set(xlabel="Thermal power [kW]", title=f"Energy demand: {MODEL.name}")
    ax.margins(x=0.2)
    fig.tight_layout()
    return fig


def plot_stream_data(res):
    """Streams handed to the pinch analysis, stacked by heat flow: cold blue, hot red."""
    fig, ax = plt.subplots(figsize=(8, 4.6))
    q, temps = {"cold": 0.0, "hot": 0.0}, []
    for s in res.stream_table():
        q0 = q[s["type"]]
        q[s["type"]] = q1 = q0 + s["Q_kW"]
        color = "tab:blue" if s["type"] == "cold" else "tab:red"
        ax.annotate("", xy=(q1, s["T_target_degC"]), xytext=(q0, s["T_supply_degC"]),
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=2, mutation_scale=14))
        ax.text((q0 + q1) / 2, (s["T_supply_degC"] + s["T_target_degC"]) / 2, "  " + s["stream"],
                color=color, fontsize=8, va="bottom")
        temps += [s["T_supply_degC"], s["T_target_degC"]]
    qmax = max(q.values()) or 1.0
    ax.set_xlim(-0.02 * qmax, 1.1 * qmax)
    ax.set_ylim(min(temps) - 10, max(temps) + 10)
    ax.set(xlabel="Heat flow [kW]", ylabel="Temperature [degC]", title="Stream data (cold: blue, hot: red)")
    fig.tight_layout()
    return fig


def plot_sensitivity(res, sweeps):
    """One-at-a-time sensitivity of the heating and cooling demand."""
    fig, axes = plt.subplots(1, len(sweeps), figsize=(6 * len(sweeps), 4), squeeze=False)
    for ax, (param, grid, xlabel) in zip(axes[0], sweeps):
        rows = []
        for x in grid:
            try:
                r = calculate({**res.inputs, param: float(x)})
                rows.append((x, r.Q_heating, r.Q_cooling))
            except ValueError:
                continue
        xs, qh, qc = zip(*rows)
        ax.plot(xs, qh, label="heating demand")
        if any(qc):
            ax.plot(xs, qc, label="cooling demand")
        ax.axvline(res.inputs[param], color="grey", ls="--", lw=0.8)
        ax.set(xlabel=xlabel, ylabel="Thermal power [kW]")
        ax.legend(fontsize=8)
    fig.suptitle("Sensitivity (one parameter varied, all others fixed)")
    fig.tight_layout()
    return fig



SWEEPS = [("T_bath", np.linspace(60, 250, 39), "Bath temperature [degC]"),
          ("product_flow_kg_h", np.linspace(0.25, 2.0, 36) * BATH["product_flow_kg_h"], "Product mass flow [kg/h]")]

In [ ]:
class Sketch:
    """Small drawing kit for the simplified P&IDs (ISA-5.1 style tags)."""

    def __init__(self, ax, lw=1.8):
        self.ax, self.lw = ax, lw

    def line(self, xs, ys, ls="-", lw=None, color="k"):
        self.ax.plot(xs, ys, ls=ls, lw=lw or self.lw, color=color, solid_capstyle="butt", zorder=2)

    def arrow(self, x0, y0, x1, y1, color="k"):
        self.ax.annotate("", xy=(x1, y1), xytext=(x0, y0), zorder=3,
                         arrowprops=dict(arrowstyle="-|>", lw=self.lw, color=color, mutation_scale=14))

    def text(self, x, y, s, **kw):
        kw.setdefault("fontsize", 9); kw.setdefault("ha", "center"); kw.setdefault("va", "center")
        self.ax.text(x, y, s, **kw)

    def hx(self, x, y, r=0.55):
        """Heat exchanger: circle with zigzag."""
        self.ax.add_patch(Circle((x, y), r, fc="white", ec="k", lw=1.6, zorder=4))
        self.ax.plot(np.linspace(x - 0.42, x + 0.42, 7), y + 0.14 * np.array([0, 1, -1, 1, -1, 1, 0]),
                     color="k", lw=1.3, zorder=5)

    def box(self, x, y, w, h, s="", fc="#eef3f8", rounded=False, fontsize=8.5):
        patch = (FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.3", fc=fc, ec="k", lw=1.6, zorder=3)
                 if rounded else Rectangle((x, y), w, h, fc=fc, ec="k", lw=1.6, zorder=3))
        self.ax.add_patch(patch)
        if s:
            self.text(x + w / 2, y + h / 2, s, fontsize=fontsize, zorder=4)

    def pump(self, x, y, tag, r=0.45):
        """Pump or fan."""
        self.ax.add_patch(Circle((x, y), r, fc="white", ec="k", lw=1.6, zorder=4))
        self.ax.add_patch(Polygon([(x - .2, y - .28), (x - .2, y + .28), (x + .3, y)], fc="k", zorder=5))
        self.text(x, y - r - 0.3, tag)

    def bubble(self, x, y, tag, num, value=None, r=0.42, below=False):
        """Instrument bubble with tag, loop number and the value it measures or controls."""
        self.ax.add_patch(Circle((x, y), r, fc="white", ec="k", lw=1.4, zorder=5))
        self.text(x, y + 0.1, tag, fontsize=8.5, fontweight="bold", zorder=6)
        self.text(x, y - 0.16, num, fontsize=8, zorder=6)
        if value is not None:
            self.text(x, y - r - 0.25 if below else y + r + 0.25, value, color="firebrick", fontweight="bold")

    def valve_h(self, x, y):
        """Control valve on a horizontal line, actuator on top."""
        for sgn in (+1, -1):
            self.ax.add_patch(Polygon([(x + sgn * .22, y + .25), (x + sgn * .22, y - .25), (x, y)],
                                      fc="white", ec="k", lw=1.4, zorder=5))
        self.ax.plot([x, x], [y, y + 0.45], color="k", lw=1.2, zorder=5)
        self.ax.add_patch(Circle((x, y + 0.65), 0.2, fc="white", ec="k", lw=1.2, zorder=5))

    def valve_v(self, x, y, side=+1):
        """Control valve on a vertical line, actuator to the side."""
        for sgn in (+1, -1):
            self.ax.add_patch(Polygon([(x - .25, y + sgn * .22), (x + .25, y + sgn * .22), (x, y)],
                                      fc="white", ec="k", lw=1.4, zorder=5))
        self.ax.plot([x, x + side * 0.45], [y, y], color="k", lw=1.2, zorder=5)
        self.ax.add_patch(Circle((x + side * 0.65, y), 0.2, fc="white", ec="k", lw=1.2, zorder=5))

    def signal(self, xs, ys):
        self.line(xs, ys, ls=":", lw=1.1)


FOOT = "Simplified P&ID (ISA-5.1 style tags), generated from the demand model - not for construction."


def new_pid(title, ymin=-1.3, ymax=9.6):
    fig, ax = plt.subplots(figsize=(16, 16 * (ymax - ymin) / 20))
    ax.set_xlim(0, 20); ax.set_ylim(ymin, ymax); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(title, fontsize=12)
    return fig, ax, Sketch(ax)


def footer(ax, s):
    ax.text(0.3, ax.get_ylim()[0] + 0.05, s, fontsize=8.5, color="dimgrey", va="bottom")

In [ ]:
def draw_pid(res):
    """Simplified P&ID of a continuously heated bath with heating coil and product conveyor."""
    v = res.inputs
    fig, ax, s = new_pid("Continuous bath heating - simplified P&ID")
    # ---- bath ---------------------------------------------------------------------------------------
    ax.add_patch(Rectangle((6.0, 1.4), 6.0, 2.8, fc="#dbe9f6", ec="none", zorder=1))
    s.line([6.0, 6.0], [4.6, 1.4], lw=2.2); s.line([6.0, 12.0], [1.4, 1.4], lw=2.2); s.line([12.0, 12.0], [1.4, 4.6], lw=2.2)
    s.line([6.0, 12.0], [4.2, 4.2], lw=1.0, color="#4c78a8")
    s.text(9.0, 3.3, f"B-101 Bath\nT_bath = {v['T_bath']:.0f} degC", zorder=4)
    # ---- product conveyor ------------------------------------------------------------------------------
    s.box(0.3, 5.0, 1.9, 1.2, f"Product in\n{v['product_flow_kg_h']:,.0f} kg/h\n{v['T_product_in']:.0f} degC")
    s.line([2.2, 6.6, 7.4, 10.6, 11.4, 13.8], [5.6, 5.6, 2.4, 2.4, 5.6, 5.6])
    s.arrow(3.6, 5.6, 4.0, 5.6); s.arrow(8.8, 2.4, 9.2, 2.4); s.arrow(12.4, 5.6, 12.8, 5.6)
    s.box(13.8, 5.0, 2.0, 1.2, f"Product out\n{v['T_bath']:.0f} degC")
    # ---- heating coil with utility -------------------------------------------------------------------
    xs = np.linspace(7.2, 10.8, 13)
    s.line(xs, 1.9 + 0.15 * np.array([(-1) ** i for i in range(13)]), lw=1.4)
    s.line([1.6, 6.9, 6.9, 7.2], [0.6, 0.6, 1.9, 1.9]); s.line([10.8, 11.1, 11.1, 15.6], [1.9, 1.9, 0.6, 0.6])
    s.arrow(2.4, 0.6, 2.8, 0.6); s.arrow(13.6, 0.6, 14.0, 0.6)
    s.valve_h(4.4, 0.6); s.text(4.4, 0.15, "TV-101", fontsize=8)
    s.text(0.3, 1.0, "steam / hot utility in" + (f", {v['T_utility_in']:.0f} degC" if v["T_utility_in"] is not None else ""),
           ha="left", fontsize=7.5)
    s.text(15.7, 0.6, "utility return" + (f", {res.T_utility_out:.1f} degC" if res.T_utility_out is not None else ""),
           ha="left", fontsize=7.5)
    s.signal([12.0, 12.58], [3.0, 3.0]); s.bubble(13.0, 3.0, "TIC", "101", f"{v['T_bath']:.0f} degC")
    # ---- losses ----------------------------------------------------------------------------------------
    for xa in (8.3, 9.7):
        s.arrow(xa, 4.3, xa, 4.9, color="firebrick")
    s.text(9.0, 5.2, f"surface losses {res.Q_surface:,.1f} kW", color="firebrick", fontsize=8.5)
    s.arrow(12.1, 2.0, 12.8, 2.0, color="firebrick")
    s.text(12.9, 2.0, f"wall losses {res.Q_trans:,.1f} kW", color="firebrick", ha="left", fontsize=8.5)
    s.text(9.0, 7.0, f"Heating demand {res.Q_heating:,.1f} kW = product {res.Q_product:,.1f} + walls {res.Q_trans:,.1f} "
           f"+ surface {res.Q_surface:,.1f} kW (Eq. 15-17)", fontsize=9.5)
    footer(ax, FOOT + " Evaporation from the bath surface is not modelled (not in Eq. 17).\n"
           f"A_surface = {v['A_surface']:g} m2, A_wall = {v['A_wall']:g} m2, U_bath = {v['U_bath']:g} W/(m2 K), "
           f"alpha = {v['alpha']:g} W/(m2 K), emissivity = {v['emissivity']:g}")
    return fig

In [ ]:
for fig in (plot_demand_breakdown(result), plot_stream_data(result), plot_sensitivity(result, SWEEPS)):
    plt.show()

fig = draw_pid(result)
fig.savefig("pid.svg", bbox_inches="tight")   # P&ID of the example, kept next to model.py
plt.show()

## 3  Validation against the manuscript
Continuous heating of metal parts in a thermal fluid bath, a referenced modelled process of a metal processing
facility, manuscript p. 18: modelled 1,142.37 kW, reference 1,196.17 kW, RMSE 53.8 kW, relative deviation 4.71 %.

**Points found in the manuscript (please check):**
1. **No inputs are given** for this case, so the 1,142.37 kW cannot be recalculated.
2. Eq. (15) is written with a mass $m$ and an energy $Q$; for a continuous bath it is used here with a mass flow.
3. $A_{wall}$ (Eq. 16) and $\varepsilon$ (Eq. 17) are not in the input list; surface evaporation is mentioned in
   the text but is not part of Eq. (17).
4. The manuscript does not state that the demand is the sum of Eq. (15)–(17); it is summed here.
5. 4.71 % refers to the model value (53.8 / 1,142.37); relative to the reference it is 4.50 %.

In [ ]:
REF = dict(Q_model=1142.37, Q_ref=1196.17, RMSE=53.8)
dev = abs(REF["Q_model"] - REF["Q_ref"])
print(f"|model - reference| = {dev:.2f} kW (manuscript RMSE {REF['RMSE']}); "
      f"{dev / REF['Q_model']:.2%} of the model value, {dev / REF['Q_ref']:.2%} of the reference")
assert abs(dev - REF["RMSE"]) < 1e-9

# ---- consistency checks -----------------------------------------------------------------------
v = result.inputs
assert abs(result.Q_product - 20000 / 3600 * 0.5 * 150) < 1e-9                                    # Eq. (15)
assert abs(result.Q_trans - 0.5 * 60 * 150 / 1000) < 1e-9                                          # Eq. (16)
rad = 0.95 * 5.67e-8 * 20 * (443.15 ** 4 - 293.15 ** 4) / 1000
assert abs(result.Q_surface - (10 * 20 * 150 / 1000 + rad)) < 1e-9                                 # Eq. (17)
assert abs(result.T_utility_out - (200 - result.Q_heating / (30000 / 3600 * 4.18))) < 1e-9          # utility balance
print("All consistency checks passed.")

## 4  Comparison with an existing implementation
Not possible: there is no earlier implementation of bath heating (the original notebook only lists it as *planned*).